In [29]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [30]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
455,I read somewhere (in a fairly panning review) ...,positive
554,"As hard as it is for me to believe, with all o...",negative
961,I don't see why all the people are giving this...,positive
460,1989 was already a year in where Eddie Murphy ...,negative
943,"""Caligula"" shares many of the same attributes ...",negative


In [31]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [32]:
df = normalize_text(df)
df.head()

,review,sentiment
455,read somewhere in fairly panning review someth...,positive
554,hard believe awful reality show past year one ...,negative
961,see people giving film negative review loved m...,positive
460,already year eddie murphy longer hot started m...,negative
943,caligula share many attribute fellini satyrico...,negative


In [33]:
df['sentiment'].value_counts()

sentiment
negative    252
positive    248
Name: count, dtype: int64

In [34]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [35]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
455,read somewhere in fairly panning review someth...,1
554,hard believe awful reality show past year one ...,0
961,see people giving film negative review loved m...,1
460,already year eddie murphy longer hot started m...,0
943,caligula share many attribute fellini satyrico...,0


In [36]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [37]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [42]:
import dagshub

mlflow.set_tracking_uri("https://dagshub.com/VikasDataSync/capstone-project.mlflow")
dagshub.init(repo_owner='VikasDataSync', repo_name='capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-05-02 22:47:10,634 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/VikasDataSync/capstone-project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "VikasDataSync/capstone-project"

2026-05-02 22:47:10,639 - INFO - Initialized MLflow to track repo "VikasDataSync/capstone-project"


Repository VikasDataSync/capstone-project initialized!

2026-05-02 22:47:10,640 - INFO - Repository VikasDataSync/capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/53ad4758a3234542b45acb58e285371d', creation_time=1777742117253, experiment_id='0', last_update_time=1777742117253, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [44]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-02 22:53:23,721 - INFO - Starting MLflow run...
2026-05-02 22:53:25,113 - INFO - Logging preprocessing parameters...
2026-05-02 22:53:26,060 - INFO - Initializing Logistic Regression model...
2026-05-02 22:53:26,060 - INFO - Fitting the model...
2026-05-02 22:53:26,075 - INFO - Model training complete.
2026-05-02 22:53:26,075 - INFO - Logging model parameters...
2026-05-02 22:53:26,404 - INFO - Making predictions...
2026-05-02 22:53:26,405 - INFO - Calculating evaluation metrics...
2026-05-02 22:53:26,414 - INFO - Logging evaluation metrics...
2026-05-02 22:53:27,682 - INFO - Saving and logging the model...
2026/05/02 22:53:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:53:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run fearless-conch-487 at: https://dagshub.com/VikasDataSync/capstone-project.mlflow/#/experiments/0/runs/2f6a388094f84100a31414124611791e
🧪 View experiment at: https://dagshub.com/VikasDataSync/capstone-project.mlflow/#/experiments/0
